# Calculadora de metas nutricionais diárias

O código nesse *notebook* puxa informações nutricionais de múltiplas fontes para cálculo da quantidade de elementos nutricionais diários listados pela TACO para uma alimentação saudável. Após computar os dados nutricionais necessários, eles serão inseridos num algoritmo de otimização com programação linear, para qualquer uma das características dos alimentos presentes.

- Fontes:
    - UNICAMP/NEPA - Tabela Brasileira de Composição de Alimentos (TACO);
    - NASEM/Health Canada - Dietary Reference Intakes, equations to estimate energy requirement;
    - Health Canada / Food and Nutrition Board - Dietary Reference Intakes tables;
    - National Academies 2019 - Dietary Reference Intakes for Sodium and Potassium;
    - WHO/FAO/UNU 2007 and FAO 2011 protein quality consultation tables.

In [15]:
import csv
from pathlib import Path
import pandas as pd

from functions import (
    calcular_necessidades,
    formatar_numero_brasileiro,
    formatar_numero_exportacao,
    limpar_rotulo,
    otimizar_dieta_lp,
    formatar_quantidade,
    formatar_tabela_exportacao,
    preparar_cobertura_humana,
    preparar_plano_humano,
    preparar_resumo_categorias,
    aplicar_meta_calorica_para_lp,
)

from IPython.display import Markdown, display  # pyright: ignore[reportUnknownVariableType]

In [16]:
DATA_DIR = Path("../data")
ID_COL = "Número do Alimento"

alimentos = pd.read_csv(DATA_DIR / "alimentos.csv")
acidos_graxos = pd.read_csv(DATA_DIR / "acidos-graxos.csv")
aminoacidos = pd.read_csv(DATA_DIR / "aminoacidos.csv")

aminoacidos["Triptofano (g)"] = pd.to_numeric(
    aminoacidos["Triptofano (g)"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce",
)

taco_completo = alimentos.merge(
    acidos_graxos, on=ID_COL, how="left", suffixes=("", "_acidos_graxos")
).merge(aminoacidos, on=ID_COL, how="left", suffixes=("", "_aminoacidos"))

output_path = DATA_DIR / "taco_completo.csv"
taco_completo.to_csv(output_path, index=False, na_rep="NA")

taco_completo.head()

try:
    tabela_bruta = taco_completo.copy()
except NameError:
    alimentos = pd.read_csv(DATA_DIR / "alimentos.csv")
    acidos_graxos = pd.read_csv(DATA_DIR / "acidos-graxos.csv")
    aminoacidos = pd.read_csv(DATA_DIR / "aminoacidos.csv")
    aminoacidos["Triptofano (g)"] = pd.to_numeric(
        aminoacidos["Triptofano (g)"].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )
    tabela_bruta = alimentos.merge(
        acidos_graxos, on=ID_COL, how="left", suffixes=("", "_acidos_graxos")
    ).merge(aminoacidos, on=ID_COL, how="left", suffixes=("", "_aminoacidos"))

colunas_repetidas = [
    coluna
    for coluna in tabela_bruta.columns
    if coluna.endswith(("_acidos_graxos", "_aminoacidos"))
    and coluna.startswith(("Categoria do alimento", "Descrição dos alimentos"))
]
tabela_limpa = tabela_bruta.drop(columns=colunas_repetidas)

tabela_formatada = tabela_limpa.copy()
colunas_numericas = tabela_formatada.select_dtypes(include="number").columns

for coluna in colunas_numericas:
    tabela_formatada[coluna] = tabela_formatada[coluna].map(formatar_numero_brasileiro)

for coluna in tabela_formatada.columns.difference(colunas_numericas):
    tabela_formatada[coluna] = tabela_formatada[coluna].replace("NA", pd.NA).fillna("")

tabela_formatada.columns = [
    limpar_rotulo(coluna) for coluna in tabela_formatada.columns
]

output_path = DATA_DIR / "taco_completo.csv"
tabela_formatada.to_csv(output_path, sep=";", index=False, quoting=csv.QUOTE_ALL)

tabela_formatada.head()

,Número do Alimento,Categoria do Alimento,Descrição dos Alimentos,Umidade,Energia (kcal),Energia (kJ),Proteína (g),Lipídeos (g),Colesterol (mg),Carboidrato (g),...,Tirosina (g),Valina (g),Arginina (g),Histidina (g),Alanina (g),Ácido Aspártico (g),Ácido Glutâmico (g),Glicina (g),Prolina (g),Serina (g)
0,1,Cereais e derivados,"Arroz, integral, cozido","70,1",124,517,"2,6",1,,"25,8",...,,,,,,,,,,
1,2,Cereais e derivados,"Arroz, integral, cru","12,2",360,1505,"7,3","1,9",,"77,5",...,,,,,,,,,,
2,3,Cereais e derivados,"Arroz, tipo 1, cozido","69,1",128,537,"2,5","0,2",,"28,1",...,,,,,,,,,,
3,4,Cereais e derivados,"Arroz, tipo 1, cru","13,2",358,1497,"7,2","0,3",,"78,8",...,,,,,,,,,,
4,5,Cereais e derivados,"Arroz, tipo 2, cozido","68,7",130,544,"2,6","0,4",,"28,2",...,,,,,,,,,,


In [17]:
perfil_usuario = {
    "sexo": "masculino",
    "idade_anos": 25,
    "altura_m": 1.84,
    "peso_kg": 136,
}

necessidades_nutricionais = calcular_necessidades(
    **perfil_usuario, # pyright: ignore[reportArgumentType]
    colunas_taco=list(tabela_formatada.columns),
)

necessidades_exportacao = necessidades_nutricionais.copy()
for coluna in ["EER usado (kcal/dia)", "Alvo", "Mínimo", "Máximo"]:
    necessidades_exportacao[coluna] = necessidades_exportacao[coluna].map(
        formatar_numero_exportacao
    )

saida_necessidades = Path("../data") / "necessidades_nutricionais_estimadas.csv"
necessidades_exportacao.to_csv(
    saida_necessidades, sep=";", index=False, quoting=csv.QUOTE_ALL
)

necessidades_nutricionais


,Estágio de vida,EER usado (kcal/dia),Nutriente,Colunas TACO usadas,Tipo,Alvo,Mínimo,Máximo,Unidade,Base científica,Observações
0,male_19_30,4098,Energia,Energia (kcal),EER,4098.0,NaN,NaN,kcal/dia,"Equação NASEM 2023 por sexo, idade, altura, pe...",
1,male_19_30,4098,Carboidrato,Carboidrato (g),RDA + AMDR,130.0,461.0,665.8,g/dia,DRI: RDA e 45-65% da energia,
2,male_19_30,4098,Proteína,Proteína (g),RDA por kg + AMDR,108.8,102.4,358.5,g/dia,"0.8 g/kg/dia, com mínimo de referência 56 g/dia",
3,male_19_30,4098,Lipídeos totais,Lipídeos (g),AMDR,NaN,91.1,159.3,g/dia,Percentual de energia vindo de gorduras totais,
4,male_19_30,4098,Fibra Alimentar,Fibra Alimentar (g),AI estimada por energia,57.4,NaN,NaN,g/dia,14 g/1000 kcal; tabela DRI também informa AI p...,AI do estágio de vida na tabela: 38 g/dia
...,...,...,...,...,...,...,...,...,...,...,...
56,male_19_30,4098,Ácido Aspártico (g),Ácido Aspártico (g),sem DRI individual,NaN,NaN,NaN,,Sem RDA/AI individual estabelecida para esta c...,"Use como dado de composição alimentar, não com..."
57,male_19_30,4098,Ácido Glutâmico (g),Ácido Glutâmico (g),sem DRI individual,NaN,NaN,NaN,,Sem RDA/AI individual estabelecida para esta c...,"Use como dado de composição alimentar, não com..."
58,male_19_30,4098,Glicina (g),Glicina (g),sem DRI individual,NaN,NaN,NaN,,Sem RDA/AI individual estabelecida para esta c...,"Use como dado de composição alimentar, não com..."
59,male_19_30,4098,Prolina (g),Prolina (g),sem DRI individual,NaN,NaN,NaN,,Sem RDA/AI individual estabelecida para esta c...,"Use como dado de composição alimentar, não com..."


In [18]:
display(
    pd.DataFrame(
        tabela_formatada["Categoria do Alimento"].unique(), columns=["Unique Values"]
    )
)

,Unique Values
0,Cereais e derivados
1,"Verduras, hortaliças e derivados"
2,Frutas e derivados
3,Gorduras e óleos
4,Pescados e frutos do mar
5,Carnes e derivados
6,Leite e derivados
7,Bebidas (alcoólicas e não alcoólicas)
8,Ovos e derivados
9,Produtos açucarados


In [19]:
# Calorie-deficit configuration for the LP optimizer.
# Keep META_CALORICA_KCAL as None to compute it from EER - DEFICIT_CALORICO_KCAL.
USAR_DEFICIT_CALORICO = True
DEFICIT_CALORICO_KCAL = 2000
META_CALORICA_KCAL = None
TOLERANCIA_CALORIAS_ABAIXO = 0.05


if USAR_DEFICIT_CALORICO:
    (
        necessidades_para_otimizacao,
        resumo_deficit_calorico,
        tolerancia_energia_acima_lp,
    ) = aplicar_meta_calorica_para_lp(
        necessidades_nutricionais,
        idade_anos=perfil_usuario["idade_anos"], # pyright: ignore[reportArgumentType]
        deficit_kcal=DEFICIT_CALORICO_KCAL,
        meta_calorica_kcal=META_CALORICA_KCAL,
        tolerancia_abaixo=TOLERANCIA_CALORIAS_ABAIXO,
    )
else:
    necessidades_para_otimizacao = necessidades_nutricionais.copy()
    resumo_deficit_calorico = {
        "EER original (kcal/dia)": float(
            necessidades_nutricionais["EER usado (kcal/dia)"].iloc[0]
        ),
        "Meta calórica (kcal/dia)": float(
            necessidades_nutricionais["EER usado (kcal/dia)"].iloc[0]
        ),
        "Déficit (kcal/dia)": 0,
        "Mínimo energético LP (kcal/dia)": float(
            necessidades_nutricionais["EER usado (kcal/dia)"].iloc[0]
        ),
        "Máximo energético LP (kcal/dia)": float(
            necessidades_nutricionais["EER usado (kcal/dia)"].iloc[0]
        )
        * 1.05,
        "Tolerância superior enviada ao LP": 0.05,
    }
    tolerancia_energia_acima_lp = 0.05

necessidades_para_otimizacao_exportacao = necessidades_para_otimizacao.copy()
for coluna in [
    "EER usado (kcal/dia)",
    "Alvo",
    "Mínimo",
    "Máximo",
    "Meta calórica para LP (kcal/dia)",
    "Déficit aplicado (kcal/dia)",
]:
    if coluna in necessidades_para_otimizacao_exportacao.columns:
        necessidades_para_otimizacao_exportacao[coluna] = (
            necessidades_para_otimizacao_exportacao[coluna].map(
                formatar_numero_exportacao
            )
        )

saida_necessidades_lp = DATA_DIR / "necessidades_para_otimizacao_lp.csv"
necessidades_para_otimizacao_exportacao.to_csv(
    saida_necessidades_lp, sep=";", index=False, quoting=csv.QUOTE_ALL
)

resumo_deficit_calorico


{'EER original (kcal/dia)': 4098.0,
 'Meta calórica (kcal/dia)': 2098.0,
 'Déficit (kcal/dia)': 2000.0,
 'Mínimo energético LP (kcal/dia)': 1993.1,
 'Máximo energético LP (kcal/dia)': 2098.0,
 'Tolerância superior enviada ao LP': 0.05263157894736836}

In [20]:
# Linear programming model:
# x[i] = number of 100 g portions of food i.
# Objective: minimize total grams of food while meeting the computed nutrient bounds.
MAX_GRAMAS_POR_ALIMENTO = 500
TOLERANCIA_ENERGIA_ACIMA = tolerancia_energia_acima_lp
MIN_GRAMAS_PARA_EXIBIR = 0.1
CATEGORIAS_PERMITIDAS = [
    "Cereais e derivados",
    "Verduras, hortaliças e derivados",
    "Frutas e derivados",
    "Gorduras e óleos",
    "Pescados e frutos do mar",
    "Carnes e derivados",
    "Ovos e derivados",
    "Miscelâneas",
    "Leguminosas e derivados",
    "Nozes e sementes",
]  # Example: ["Cereais e derivados", "Carnes e derivados"]
TERMOS_EXCLUIDOS = None  # Example: ["café, pó", "gelatina", "fermento"]

COLUNAS_IDENTIFICACAO = [
    "Número do Alimento",
    "Categoria do Alimento",
    "Descrição dos Alimentos",
]

try:
    tabela_base_lp = tabela_limpa.copy()
except NameError:
    tabela_base_lp = pd.read_csv(DATA_DIR / "taco_completo.csv", sep=";")

try:
    necessidades_lp = necessidades_para_otimizacao.copy()
except NameError:
    try:
        necessidades_lp = necessidades_nutricionais.copy()
    except NameError:
        necessidades_lp = pd.read_csv(
            DATA_DIR / "necessidades_para_otimizacao_lp.csv", sep=";"
        )

resumo_otimizacao, dieta_otimizada, cobertura_dieta_otimizada = otimizar_dieta_lp(
    tabela_base=tabela_base_lp,
    necessidades=necessidades_lp,
    max_gramas_por_alimento=MAX_GRAMAS_POR_ALIMENTO,
    tolerancia_energia_acima=TOLERANCIA_ENERGIA_ACIMA,
    categorias_permitidas=CATEGORIAS_PERMITIDAS,
    termos_excluidos=TERMOS_EXCLUIDOS,
    min_gramas_para_exibir=MIN_GRAMAS_PARA_EXIBIR,
)

for caminho, tabela in [
    (DATA_DIR / "dieta_otimizada_lp.csv", dieta_otimizada),
    (DATA_DIR / "cobertura_dieta_otimizada_lp.csv", cobertura_dieta_otimizada),
]:
    tabela_exportacao = tabela.copy()
    for coluna in tabela_exportacao.select_dtypes(include="number").columns:
        tabela_exportacao[coluna] = tabela_exportacao[coluna].map(
            formatar_numero_exportacao
        )
    tabela_exportacao.to_csv(caminho, sep=";", index=False, quoting=csv.QUOTE_ALL)

dieta_otimizada

,Número do Alimento,Categoria do Alimento,Descrição dos Alimentos,Quantidade (g),Porções de 100 g,Energia (kcal) no plano,Proteína (g) no plano,Carboidrato (g) no plano,Lipídeos (g) no plano,Fibra Alimentar (g) no plano,Sódio (mg) no plano
0,147,"Verduras, hortaliças e derivados","Quiabo, cru",500.000000,5.000000,150.000000,9.500000,32.000000,1.500000,23.000000,5.000000
1,115,"Verduras, hortaliças e derivados","Couve, manteiga, crua",312.188610,3.121886,84.290925,9.053470,13.424110,1.560943,9.677847,18.731317
2,83,"Verduras, hortaliças e derivados","Alho-poró, cru",299.390293,2.993903,95.804894,4.191464,20.657930,0.299390,7.484757,5.987806
3,323,Carnes e derivados,Apresuntado,235.056342,2.350563,303.222681,31.732606,6.816634,15.748775,0.000000,2216.581305
4,580,Leguminosas e derivados,"Pé-de-moleque, amendoim",211.958578,2.119586,1066.151647,27.978532,115.941342,59.348402,7.206592,33.913372
5,568,Leguminosas e derivados,"Feijão, preto, cru",82.081298,0.820813,265.943404,17.483316,48.263803,0.984976,17.893723,0.000008
6,514,Miscelâneas,"Fermento, biológico, levedura, tablete",48.476837,0.484768,43.629153,8.241062,3.732716,0.727153,2.036027,19.390735
7,594,Nozes e sementes,"Linhaça, semente",4.393965,0.043940,21.750127,0.619549,1.902587,1.419251,1.471978,0.395457


In [21]:
dieta_base_humana = dieta_otimizada.copy()
cobertura_base_humana = cobertura_dieta_otimizada.copy()

plano_alimentar_legivel = preparar_plano_humano(dieta_base_humana)
resumo_por_categoria = preparar_resumo_categorias(plano_alimentar_legivel)
cobertura_legivel = preparar_cobertura_humana(cobertura_base_humana)

arquivos_legiveis = {
    DATA_DIR / "plano_alimentar_legivel.csv": plano_alimentar_legivel,
    DATA_DIR / "plano_alimentar_resumo_categorias.csv": resumo_por_categoria,
    DATA_DIR / "plano_alimentar_cobertura_legivel.csv": cobertura_legivel,
}

for caminho, tabela in arquivos_legiveis.items():
    formatar_tabela_exportacao(tabela).to_csv(
        caminho, sep=";", index=False, quoting=csv.QUOTE_ALL
    )

total_diario = formatar_quantidade(
    plano_alimentar_legivel["Quantidade diária (g)"].sum()
)
total_semanal = formatar_quantidade(
    plano_alimentar_legivel["Quantidade semanal (g)"].sum()
)
restricoes_ok = int(cobertura_legivel["Status"].eq("OK").sum())
total_restricoes = len(cobertura_legivel)

linhas_markdown = [
    "# Plano alimentar otimizado",
    "",
    f"- Total aproximado: {total_diario} por dia",
    f"- Equivalente semanal: {total_semanal} por semana",
    f"- Alimentos no plano: {len(plano_alimentar_legivel)}",
    f"- Restrições nutricionais atendidas: {restricoes_ok}/{total_restricoes}",
]

try:
    linhas_markdown.extend(
        [
            f"- Meta calórica: {formatar_quantidade(resumo_deficit_calorico['Meta calórica (kcal/dia)'], 'kcal')} por dia",  # noqa: E501
            f"- Déficit aplicado: {formatar_quantidade(resumo_deficit_calorico['Déficit (kcal/dia)'], 'kcal')} por dia",  # noqa: E501
        ]
    )
except NameError:
    pass

linhas_markdown.extend(["", "## Alimentos", ""])

for _, linha in plano_alimentar_legivel.iterrows():
    observacao = (
        f" ({linha['Observação prática']})" if linha["Observação prática"] else ""
    )
    linhas_markdown.append(
        f"- {linha['Descrição dos Alimentos']}: {linha['Formato sugerido']}{observacao}"
    )

linhas_markdown.extend(
    [
        "",
        "## Nota",
        "",
        (
            "Este plano minimiza massa total de alimentos, não sabor, "
            "variedade, custo, saciedade ou adequação culinária. Use os "
            "campos CATEGORIAS_PERMITIDAS e TERMOS_EXCLUIDOS na célula "
            "de otimização para deixar o resultado mais parecido com "
            "comida de verdade."
        ),
    ]
)

relatorio_markdown = "\n".join(linhas_markdown)
(DATA_DIR / "plano_alimentar_legivel.md").write_text(
    relatorio_markdown, encoding="utf-8"
)

display(Markdown(relatorio_markdown))
plano_alimentar_legivel

# Plano alimentar otimizado

- Total aproximado: 1690 g por dia
- Equivalente semanal: 11855 g por semana
- Alimentos no plano: 8
- Restrições nutricionais atendidas: 33/33
- Meta calórica: 2100 kcal por dia
- Déficit aplicado: 2000 kcal por dia

## Alimentos

- Quiabo, cru: 500 g por dia (porção diária alta; peso da TACO pode mudar após preparo)
- Couve, manteiga, crua: 310 g por dia (porção diária alta; peso da TACO pode mudar após preparo)
- Alho-poró, cru: 300 g por dia (porção diária alta; peso da TACO pode mudar após preparo)
- Apresuntado: 235 g por dia
- Pé-de-moleque, amendoim: 210 g por dia
- Feijão, preto, cru: 82 g por dia (peso da TACO pode mudar após preparo)
- Fermento, biológico, levedura, tablete: 340 g por semana (~48 g/dia)
- Linhaça, semente: 31 g por semana (microquantidade; mais fácil planejar por semana; peso da TACO pode mudar após preparo)

## Nota

Este plano minimiza massa total de alimentos, não sabor, variedade, custo, saciedade ou adequação culinária. Use os campos CATEGORIAS_PERMITIDAS e TERMOS_EXCLUIDOS na célula de otimização para deixar o resultado mais parecido com comida de verdade.

,Descrição dos Alimentos,Categoria do Alimento,Formato sugerido,Quantidade diária (g),Quantidade semanal (g),Quantidade mensal (g),Energia (kcal) no plano,Proteína (g) no plano,Carboidrato (g) no plano,Lipídeos (g) no plano,Fibra Alimentar (g) no plano,Sódio (mg) no plano,Observação prática
0,"Quiabo, cru","Verduras, hortaliças e derivados",500 g por dia,500.0,3500,15000,150.0,9.5,32.0,1.5,23.0,5.0,porção diária alta; peso da TACO pode mudar ap...
1,"Couve, manteiga, crua","Verduras, hortaliças e derivados",310 g por dia,310.0,2185,9365,84.0,9.1,13.4,1.6,9.7,19.0,porção diária alta; peso da TACO pode mudar ap...
2,"Alho-poró, cru","Verduras, hortaliças e derivados",300 g por dia,300.0,2095,8980,96.0,4.2,20.7,0.3,7.5,6.0,porção diária alta; peso da TACO pode mudar ap...
3,Apresuntado,Carnes e derivados,235 g por dia,235.0,1645,7050,303.0,31.7,6.8,15.7,0.0,2217.0,
4,"Pé-de-moleque, amendoim",Leguminosas e derivados,210 g por dia,210.0,1485,6360,1066.0,28.0,115.9,59.3,7.2,34.0,
5,"Feijão, preto, cru",Leguminosas e derivados,82 g por dia,82.0,575,2460,266.0,17.5,48.3,1.0,17.9,0.0,peso da TACO pode mudar após preparo
6,"Fermento, biológico, levedura, tablete",Miscelâneas,340 g por semana (~48 g/dia),48.0,340,1455,44.0,8.2,3.7,0.7,2.0,19.0,
7,"Linhaça, semente",Nozes e sementes,31 g por semana,4.4,31,130,22.0,0.6,1.9,1.4,1.5,0.0,microquantidade; mais fácil planejar por seman...
